# Day 8 — Probability distributions visual laboratory

This notebook is a compact interactive companion to the executable scripts. All observations are synthetic and reproducible.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from distribution_utils import SEED, calculate_empirical_statistics, create_normal_sigma_surface

rng = np.random.default_rng(SEED)
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## The six families

The table emphasizes support and mechanism before visual shape.

In [ ]:
pd.DataFrame(
    [
        ("Bernoulli", "{0, 1}", "One binary outcome"),
        ("Binomial", "0, …, n", "Successes in fixed trials"),
        ("Poisson", "0, 1, 2, …", "Events per exposure"),
        ("Exponential", "t ≥ 0", "Waiting time in a homogeneous Poisson process"),
        ("Normal", "All real values", "Additive symmetric variation"),
        ("Log-normal", "y > 0", "Positive multiplicative variation"),
    ],
    columns=["Distribution", "Support", "Mechanism"],
)

## Theory versus simulation

Increase `sample_size` and observe empirical moments move toward their theoretical values.

In [ ]:
sample_size = 20_000
configs = [
    ("Bernoulli", rng.binomial(1, 0.7, sample_size), 0.7, 0.7 * 0.3),
    ("Binomial", rng.binomial(20, 0.35, sample_size), 7.0, 20 * 0.35 * 0.65),
    ("Poisson", rng.poisson(4.0, sample_size), 4.0, 4.0),
    ("Exponential", rng.exponential(2.0, sample_size), 2.0, 4.0),
    ("Normal", rng.normal(10.0, 2.0, sample_size), 10.0, 4.0),
    (
        "Log-normal",
        rng.lognormal(2.0, 0.6, sample_size),
        np.exp(2.0 + 0.6**2 / 2),
        (np.exp(0.6**2) - 1) * np.exp(2 * 2.0 + 0.6**2),
    ),
]

rows = []
for name, sample, theoretical_mean, theoretical_variance in configs:
    empirical = calculate_empirical_statistics(sample)
    rows.append(
        {
            "Distribution": name,
            "Theoretical mean": theoretical_mean,
            "Empirical mean": empirical.mean,
            "Theoretical variance": theoretical_variance,
            "Empirical variance": empirical.variance,
            "Empirical skewness": empirical.skewness,
        }
    )
pd.DataFrame(rows).round(4)

## Binomial to Poisson approximation

Keep $\lambda=np=5$ fixed while increasing $n$ and decreasing $p$.

In [ ]:
support = np.arange(0, 18)
poisson_mass = stats.poisson.pmf(support, 5)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for axis, n in zip(axes, [10, 100, 1000], strict=True):
    p = 5 / n
    binomial_mass = stats.binom.pmf(support, n, p)
    distance = np.abs(binomial_mass - poisson_mass).sum()
    axis.bar(support, binomial_mass, alpha=0.55, label="Binomial")
    axis.plot(support, poisson_mass, "o-", label="Poisson")
    axis.set_title(f"n={n}, p={p:.4f}\nL1 distance={distance:.4f}")
    axis.set_xlabel("Count")
axes[0].set_ylabel("Probability mass")
axes[-1].legend()
fig.tight_layout()

## Tail behavior: median-matched candidates

Equal central values do not imply equal p95 or p99 latency.

In [ ]:
normal_latency = rng.normal(1.0, 0.22, 30_000)
lognormal_latency = rng.lognormal(0.0, 0.55, 30_000)
normal_latency /= np.median(normal_latency)
lognormal_latency /= np.median(lognormal_latency)

pd.DataFrame(
    {
        "Metric": ["mean", "p50", "p90", "p95", "p99"],
        "Normal candidate": [
            normal_latency.mean(),
            *np.quantile(normal_latency, [0.50, 0.90, 0.95, 0.99]),
        ],
        "Log-normal candidate": [
            lognormal_latency.mean(),
            *np.quantile(lognormal_latency, [0.50, 0.90, 0.95, 0.99]),
        ],
    }
).round(3)

## Meaningful 3D parameter surface

Rotate the surface to see how increasing $\sigma$ makes a Normal density wider and lower while preserving total probability.

In [ ]:
create_normal_sigma_surface(mu=0.0).show()